# Silver Layer — Cleaning & Validation
## NYC Yellow Taxi — January 2024

Reads from the Bronze raw table, applies data quality rules,
adds derived columns, and saves a clean Delta table to Silver.

**Source:** `nyc_taxi.bronze.yellow_trips_raw`  
**Target:** `nyc_taxi.silver.yellow_trips_clean`

In [0]:
from pyspark.sql import functions as F

df = spark.table("nyc_taxi.bronze.yellow_trips_raw")
df.count()

2964624

In [0]:
# Data cleaning

# Scope to January 2024 — source data may contain records outside this window
df1 = df.filter("tpep_pickup_datetime >= '2024-01-01' AND tpep_pickup_datetime < '2024-02-01'")

# Remove rows with negative trip distance
df1 = df1.filter("trip_distance > 0")

# Remove rows with no realistic duration e.g. more than 300 mins
df1 = df1.withColumn("trip_duration_minutes", 
    (F.unix_timestamp(F.col("tpep_dropoff_datetime").cast("timestamp")) - F.unix_timestamp(F.col("tpep_pickup_datetime").cast("timestamp")) )/ 60)
df1 = df1.filter("trip_duration_minutes > 0 AND trip_duration_minutes < 300 AND tpep_dropoff_datetime > tpep_pickup_datetime ")

# Flag records with data quality issues instead of dropping them.
# Allows downstream layers to filter by severity while preserving full audit trail.
df1 = df1.withColumn("dq_flag", F.expr("""CASE WHEN passenger_count = 0 AND trip_duration_minutes > 0 AND total_amount > 0 THEN 'missing_passenger'
                                                 WHEN passenger_count = 0 AND trip_duration_minutes > 0 AND total_amount = 0 THEN 'no_passenger_trip' 
                                                 WHEN passenger_count = 0 AND trip_duration_minutes = 0 THEN 'invalid_trip'
                                                 WHEN passenger_count > 0 AND trip_duration_minutes = 0 AND total_amount = 0 THEN 'zero_passenger_zero_amount'
                                         ELSE NULL END"""))

# Keep trips that have at least a valid passenger count or a non-zero charge
df1 = df1.filter("passenger_count > 0 OR total_amount > 0")

# Remove rows with negative fare amount
df1 = df1.filter("fare_amount > 0")

# Remove rows with missing location coordinates
df1 = df1.filter("PULocationID IS NOT NULL AND DOLocationID IS NOT NULL")

# Remove rows with negative tip amount
df1 = df1.filter("tip_amount >= 0")

# Remove rows with negative toll amount
df1 = df1.filter("tolls_amount >= 0")

# Remove rows with negative improvement surcharge
df1 = df1.filter("improvement_surcharge >= 0")

# Remove rows with negative total amount
df1 = df1.filter("total_amount >= 0")

# Remove rows with negative congestion surcharge
df1 = df1.filter("congestion_surcharge >= 0")

# Remove rows with negative airport fee
df1 = df1.filter("Airport_fee >= 0")

In [0]:
# Add time-based columns for Gold aggregations
df1 = df1.withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
df1 = df1.withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))
df1 = df1.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

In [0]:
df1.write.format("delta").mode("overwrite").saveAsTable("nyc_taxi.silver.yellow_trips_clean")

In [0]:
# Raw data row count
print(f"Number of rows in raw data: {df.count()}")
# Cleaned data row count
print(f"Number of rows in cleaned data: {df1.count()}")

Number of rows in raw data: 2964624
Number of rows in cleaned data: 2752609
